# 노드/엣지 연결 패턴

이 노트북에서 다루는 4가지 연결 패턴:

1. **순차 연결** — `step_1 → step_2 → step_3` (가장 기본)
2. **`add_sequence`** — 순차를 한 줄로
3. **병렬 (fan-out / fan-in)** — `a → {b, c} → d`
4. **조건부 병렬** — 라우터가 다음에 실행할 노드를 **리스트로 반환**

병렬이 등장하는 순간부터는 **여러 노드가 같은 State 필드를 동시에 갱신**하므로 Reducer가 사실상 필수다.
(예: `Annotated[list, operator.add]`)

## 1. 순차 연결 (기본)

`add_node(fn)` 으로 등록하면 **함수명이 그대로 노드 이름**이 된다. 그래서 엣지 추가 시 `"step_1"` 같은 문자열로 지정 가능.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    value_1: str
    value_2: int

def step_1(state: State):
    return {"value_1": state["value_1"]}

def step_2(state: State):
    return {"value_1": f"{state['value_1']} b"}

def step_3(state: State):
    return {"value_2": 10}

gb = StateGraph(State)
gb.add_node(step_1)
gb.add_node(step_2)
gb.add_node(step_3)

gb.add_edge(START, "step_1")
gb.add_edge("step_1", "step_2")
gb.add_edge("step_2", "step_3")
gb.add_edge("step_3", END)

graph = gb.compile()
graph.invoke({"value_1": "a", "value_2": 0})

In [ ]:
print(graph.get_graph().draw_mermaid())

## 2. `add_sequence` — 순차를 한 줄로

여러 노드를 차례로 잇는다면 `add_sequence([...])` 로 한 번에 표현할 수 있다.
내부적으로 자동으로 엣지를 깐다 (`step_1 → step_2 → step_3`).
단, `START` 와의 연결은 여전히 수동으로 해야 한다.

In [ ]:
gb = StateGraph(State).add_sequence([step_1, step_2, step_3])
gb.add_edge(START, "step_1")
# add_sequence 가 step_3 -> END 까지는 만들어주지 않으므로 명시
gb.add_edge("step_3", END)

graph = gb.compile()
graph.invoke({"value_1": "c", "value_2": 0})

## 3. 병렬 연결 (fan-out → fan-in)

```
       ┌── b ──┐
a ─────┤        ├──> d
       └── c ──┘
```

한 노드(`a`) 가 끝난 뒤 **두 노드(`b`, `c`) 가 병렬로 실행** 되고,
둘 다 끝나야 합류 노드(`d`) 가 실행된다.

여러 노드가 같은 State 필드(`aggregate`) 를 동시에 갱신하므로 **Reducer 필수** — `operator.add` 로 리스트를 합친다.

In [ ]:
import operator
from typing import Annotated

class ParState(TypedDict):
    aggregate: Annotated[list, operator.add]

def a(s):
    print(f'Adding "A" to {s["aggregate"]}')
    return {"aggregate": ["A"]}

def b(s):
    print(f'Adding "B" to {s["aggregate"]}')
    return {"aggregate": ["B"]}

def c(s):
    print(f'Adding "C" to {s["aggregate"]}')
    return {"aggregate": ["C"]}

def d(s):
    print(f'Adding "D" to {s["aggregate"]}')
    return {"aggregate": ["D"]}

gb = StateGraph(ParState)
gb.add_node(a); gb.add_node(b); gb.add_node(c); gb.add_node(d)

gb.add_edge(START, "a")
gb.add_edge("a", "b")          # a -> b
gb.add_edge("a", "c")          # a -> c   (둘이 병렬)
gb.add_edge("b", "d")
gb.add_edge("c", "d")          # b, c 모두 끝나야 d 실행 (자동 합류)
gb.add_edge("d", END)

graph = gb.compile()
graph.invoke({"aggregate": []})

In [ ]:
print(graph.get_graph().draw_mermaid())

## 4. 조건부 병렬 — 라우터가 "리스트" 를 반환

조건부 엣지의 라우팅 함수는 보통 다음 노드 이름 하나를 반환한다.
하지만 **리스트로 반환하면** 그 노드들이 **모두 병렬 실행**된다.

아래 예시는 `which` 값에 따라 `[b, c]` 혹은 `[c, d]` 를 동시에 실행하고, 마지막에 `e` 로 합류한다.

```
        ┌─ b ─┐
a ──────┤      ├──> e
        ├─ c ─┤
        └─ d ─┘   (which 에 따라 둘만 선택)
```

`add_conditional_edges(노드, 라우터, [가능한 다음 노드들])` 의 세 번째 인자는 시각화에서 어떤 분기들이 가능한지 알려주는 용도.

In [ ]:
from typing import Sequence

class CondParState(TypedDict):
    aggregate: Annotated[list, operator.add]
    which: str

def e(s):
    print(f'Adding "E" to {s["aggregate"]}')
    return {"aggregate": ["E"]}

def route_bc_or_cd(state: CondParState) -> Sequence[str]:
    return ["c", "d"] if state["which"] == "cd" else ["b", "c"]

intermediates = ["b", "c", "d"]

gb = StateGraph(CondParState)
gb.add_node(a); gb.add_node(b); gb.add_node(c); gb.add_node(d); gb.add_node(e)

gb.add_edge(START, "a")
gb.add_conditional_edges("a", route_bc_or_cd, intermediates)
for n in intermediates:
    gb.add_edge(n, "e")        # 어떤 분기든 e 로 합류
gb.add_edge("e", END)

graph = gb.compile()

print("which=bc →", graph.invoke({"aggregate": [], "which": "bc"}))
print("which=cd →", graph.invoke({"aggregate": [], "which": "cd"}))

In [ ]:
print(graph.get_graph().draw_mermaid())

## 정리

| 패턴 | 핵심 API |
|---|---|
| 순차 | `add_edge(src, dst)` 반복 |
| 순차 (단축) | `add_sequence([fn1, fn2, ...])` |
| 병렬 fan-out/in | 같은 src에서 여러 dst로 `add_edge` |
| 조건부 병렬 | 라우터가 `list[str]` 반환 + `add_conditional_edges(src, router, [...])` |

**병렬이 등장하면 Reducer가 필수**다.
여러 노드가 같은 필드를 동시에 갱신할 때 "합치는 규칙" 이 없으면 LangGraph 가 에러를 낸다.

다음: `04_loops_and_routing.ipynb` — 루프, recursion_limit, 더 복잡한 조건부 라우팅, 사용자 입력 루프.